In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

def _get_X(adata, layer=None, use_raw=False):
    if use_raw:
        if adata.raw is None:
            raise ValueError("use_raw=True but adata.raw is None")
        X = adata.raw.X
        var_names = np.asarray(adata.raw.var_names)
    else:
        X = adata.layers[layer] if layer is not None else adata.X
        var_names = np.asarray(adata.var_names)
    return X, var_names

def dotplot_stats(
    adata,
    genes,
    groupby="cell_type",
    *,
    layer=None,
    use_raw=False,
    expr_threshold=0.0,  # "expressed" means > expr_threshold
):
    """
    Returns:
      avg_expr: (n_genes, n_groups) mean expression
      pct_expr: (n_genes, n_groups) fraction expressed
      groups: list of group names (column order)
      genes_use: list of genes actually used (row order)
    """
    X, var_names = _get_X(adata, layer=layer, use_raw=use_raw)

    groups = pd.Index(adata.obs[groupby].astype(str))
    group_levels = groups.unique().tolist()

    # map genes -> indices
    gene_to_idx = {g: i for i, g in enumerate(var_names)}
    genes_use = [g for g in genes if g in gene_to_idx]
    if len(genes_use) == 0:
        raise ValueError("None of the requested genes were found in adata.var_names (or adata.raw.var_names).")

    g_idx = np.array([gene_to_idx[g] for g in genes_use], dtype=int)

    # slice to cells x genes_sub
    if sparse.issparse(X):
        Xg = X[:, g_idx].tocsr()
    else:
        Xg = np.asarray(X)[:, g_idx]

    avg_expr = np.zeros((len(genes_use), len(group_levels)), dtype=float)
    pct_expr = np.zeros((len(genes_use), len(group_levels)), dtype=float)

    for j, lvl in enumerate(group_levels):
        cell_mask = (groups.values == lvl)
        n = int(cell_mask.sum())
        if n == 0:
            continue

        if sparse.issparse(Xg):
            Xsub = Xg[cell_mask, :]  # (n_cells, n_genes)
            # mean
            avg_expr[:, j] = np.asarray(Xsub.mean(axis=0)).ravel()
            # pct expressed
            pct_expr[:, j] = np.asarray((Xsub > expr_threshold).mean(axis=0)).ravel()
        else:
            Xsub = Xg[cell_mask, :]
            avg_expr[:, j] = Xsub.mean(axis=0)
            pct_expr[:, j] = (Xsub > expr_threshold).mean(axis=0)

    return avg_expr, pct_expr, group_levels, genes_use

def scale_rows_01(mat):
    """Scale each row to [0,1] (per gene scaling across groups)."""
    mat = mat.astype(float)
    lo = mat.min(axis=1, keepdims=True)
    hi = mat.max(axis=1, keepdims=True)
    den = np.where((hi - lo) == 0, 1.0, (hi - lo))
    return (mat - lo) / den

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

def two_panel_dotplot(
    adata_sc,
    adata_sp,
    genes,
    *,
    groupby="cell_type",
    celltype_order=None,   # list of cell types in desired x-order
    gene_group_map=None,   # dict gene -> group_name (for the left colored strip)
    group_colors=None,     # dict group_name -> color
    layer_sc=None,
    layer_sp=None,
    use_raw_sc=False,
    use_raw_sp=False,
    expr_threshold=0.0,
    figsize=(12, 6),
    dpi=150,
    size_min=10,
    size_max=200,
    cmap="Purples",        # close to your example
):
    # --- compute stats ---
    avg_sc, pct_sc, groups_sc, genes_use = dotplot_stats(
        adata_sc, genes, groupby=groupby, layer=layer_sc, use_raw=use_raw_sc, expr_threshold=expr_threshold
    )
    avg_sp, pct_sp, groups_sp, _ = dotplot_stats(
        adata_sp, genes_use, groupby=groupby, layer=layer_sp, use_raw=use_raw_sp, expr_threshold=expr_threshold
    )

    # enforce same group order
    if celltype_order is None:
        # use sc groups order but intersect with spatial
        celltype_order = [g for g in groups_sc if g in set(groups_sp)]
    else:
        celltype_order = [g for g in celltype_order if g in set(groups_sc) and g in set(groups_sp)]

    idx_sc = [groups_sc.index(ct) for ct in celltype_order]
    idx_sp = [groups_sp.index(ct) for ct in celltype_order]

    avg_sc = avg_sc[:, idx_sc]
    pct_sc = pct_sc[:, idx_sc]
    avg_sp = avg_sp[:, idx_sp]
    pct_sp = pct_sp[:, idx_sp]

    # scale avg expression per gene to [0,1]
    col_sc = scale_rows_01(avg_sc)
    col_sp = scale_rows_01(avg_sp)

    # sizes from percent expressed
    def size_map(p):
        p = np.clip(p, 0, 1)
        return size_min + (size_max - size_min) * p

    S_sc = size_map(pct_sc)
    S_sp = size_map(pct_sp)

    # --- figure layout ---
    fig = plt.figure(figsize=figsize, dpi=dpi)
    gs = fig.add_gridspec(nrows=1, ncols=3, width_ratios=[0.06, 1, 1], wspace=0.25)

    ax_strip = fig.add_subplot(gs[0, 0])
    ax_sc = fig.add_subplot(gs[0, 1])
    ax_sp = fig.add_subplot(gs[0, 2])

    # --- left gene-group strip ---
    ax_strip.set_xlim(0, 1)
    ax_strip.set_ylim(-0.5, len(genes_use) - 0.5)
    ax_strip.axis("off")

    if gene_group_map is not None:
        # draw a colored rectangle per gene row (simple + effective)
        for i, g in enumerate(genes_use):
            grp = gene_group_map.get(g, None)
            color = None
            if grp is not None and group_colors is not None:
                color = group_colors.get(grp, None)
            if color is None:
                color = "lightgray"
            ax_strip.add_patch(plt.Rectangle((0, i - 0.5), 1, 1, color=color, ec="none"))

    # common plotting routine
    def _dotpanel(ax, colors, sizes, title):
        n_genes, n_ct = colors.shape
        xs, ys, cs, ss = [], [], [], []
        for i in range(n_genes):
            for j in range(n_ct):
                xs.append(j)
                ys.append(i)
                cs.append(colors[i, j])
                ss.append(sizes[i, j])

        sc = ax.scatter(xs, ys, c=cs, s=ss, cmap=cmap, vmin=0, vmax=1, edgecolors="none")
        ax.set_title(title, fontsize=14)

        ax.set_xticks(range(n_ct))
        ax.set_xticklabels(celltype_order, rotation=90)
        ax.set_yticks(range(n_genes))
        ax.set_yticklabels(genes_use, fontstyle="italic")  # looks like the paper

        ax.set_xlim(-0.5, n_ct - 0.5)
        ax.set_ylim(n_genes - 0.5, -0.5)  # invert y so first gene is at top

        ax.grid(True, which="major", color="#dddddd", linewidth=0.8)
        ax.set_axisbelow(True)
        return sc

    sc1 = _dotpanel(ax_sc, col_sc, S_sc, "scRNA data")
    _ = _dotpanel(ax_sp, col_sp, S_sp, "Spatial data")

    ax_sc.set_ylabel("Marker genes", fontsize=12)
    ax_sp.set_ylabel("")

    # --- colorbar: scaled average expression ---
    cbar = fig.colorbar(sc1, ax=[ax_sc, ax_sp], fraction=0.03, pad=0.02)
    cbar.set_label("Scaled average expression", rotation=90)

    # --- size legend: percent expressed ---
    # Use a few reference sizes
    ref = np.array([0.25, 0.50, 0.75])
    handles = [
        plt.Line2D([], [], marker="o", linestyle="", color="black",
                   markersize=np.sqrt(size_map(r))/2)  # approximate
        for r in ref
    ]
    labels = [f"{int(r*100)}" for r in ref]
    ax_sp.legend(handles, labels, title="Percent expressed",
                 loc="lower center", bbox_to_anchor=(0.5, -0.25),
                 ncol=3, frameon=False)

    return fig, (ax_strip, ax_sc, ax_sp)